# 05. PII Detection — Azure AI Language (`recognize_pii_entities`)

This notebook calls the **dedicated Azure AI Language** service directly via the `azure-ai-textanalytics` SDK
— the first notebook in this folder that isn't an LLM prompt wrapper. It uses `TextAnalyticsClient` and the
`recognize_pii_entities` prebuilt model to find personally identifiable information (names, emails, phone
numbers, etc.) in a sample support message, and prints both the detected entities and a **redacted** version
of the text with PII masked out.

**Difficulty:** Beginner


## Prerequisites

**Python packages:**
- `azure-ai-textanalytics` — **not yet in the repo's root `requirements.txt`**, install with:
  ```
  pip install azure-ai-textanalytics
  ```
- `python-dotenv` (already in `requirements.txt`)

**Azure resources required:**
- An Azure AI Language resource (or a multi-service Azure AI Foundry resource that exposes Language) with a
  key and endpoint

**Environment variables** (add to root `.env`):
```
AZURE_LANGUAGE_ENDPOINT=https://<your-language-resource>.services.ai.azure.com/
AZURE_LANGUAGE_KEY=<your-language-resource-key>
```


## What You'll Learn

- How to construct a `TextAnalyticsClient` with key-based auth (`AzureKeyCredential`)
- How to call `recognize_pii_entities()` and interpret the response (`entities`, `redacted_text`,
  `confidence_score`)
- How PII detection differs from generic Named Entity Recognition (notebook 07) even though both use the same
  client class
- How to handle per-document errors in a batch response (`doc.is_error`)


### Step 1 — Create the Text Analytics client

`TextAnalyticsClient` is the single entry point for *all* Azure AI Language prebuilt text operations —
sentiment, key phrases, NER, PII, language detection, and more all go through this same client class, just
with different method calls. Auth here uses `AzureKeyCredential` (a resource key), the simpler alternative to
Entra ID/`DefaultAzureCredential` used in the Foundry notebooks.

💡 **Exam tip:** AI-102 expects you to know both auth patterns for Cognitive Services resources: **key-based**
(`AzureKeyCredential`, simple, key rotation is your responsibility) and **Entra ID / managed identity**
(`DefaultAzureCredential`, no long-lived secret, preferred for production). `TextAnalyticsClient` supports
both.

🔄 **Alternatives:** Call the Language REST API directly with `requests` (as notebook 09 does for Translator)
instead of the SDK — useful in environments where you can't install the Python SDK, at the cost of manually
handling request/response shapes.


In [ ]:
import os
from dotenv import load_dotenv
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

load_dotenv()

endpoint = os.environ["AZURE_LANGUAGE_ENDPOINT"]
api_key = os.environ["AZURE_LANGUAGE_KEY"]

client = TextAnalyticsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key),
)

### Step 2 — Detect PII entities in a batch of documents

`documents` is a list of strings — the Language service's synchronous APIs are batch-oriented by design: you
pass a list (even if it has one element) and get back a list of per-document results in the same order.

`recognize_pii_entities(documents, language="en")` runs the prebuilt PII detection model, which looks for
categories like `Person`, `PhoneNumber`, `Email`, `Address`, and many others depending on region/locale
support.

💡 **Exam tip:** Know that Language service calls take a **language code** (`language="en"`) per call (or per
document) — this tells the service which language model to run, distinct from *detecting* the language, which
is a separate operation (`detect_language`, notebook 06). If you don't know the language ahead of time, detect
it first, then pass the detected code into PII/NER/sentiment calls for best accuracy.

🔄 **Alternatives:** PII detection is also available for **conversation transcripts** via a separate
conversational PII operation in the Language service, tuned for multi-turn dialogue rather than single
documents — relevant if you're redacting call-center transcripts rather than support tickets.


In [ ]:
documents = [
    "Hi, this is Sarah Chen from Acme Logistics. You can reach me at "
    "sarah.chen@acmelogistics.com or call 312-555-1234 regarding ticket TKT-1042.",
]

response = client.recognize_pii_entities(documents, language="en")
results = [doc for doc in response if not doc.is_error]

### Step 3 — Print the redacted text and the detected entities

Every successful result exposes `doc.redacted_text` — the original text with each detected PII span replaced
by asterisks — plus `doc.entities`, a list of `PiiEntity` objects each with `.category`, `.text`, and
`.confidence_score` (0.0–1.0).

The `if not doc.is_error` filter on the previous step is important: in a batch call, Azure AI Language can
successfully process some documents and fail others (e.g. due to per-document errors like unsupported
language) — `is_error` on each result tells you which is which, rather than the whole batch failing at once.

💡 **Exam tip:** `confidence_score` is a per-entity float the Language service returns for nearly every
recognition operation (PII, NER, sentiment) — AI-102 expects you to know it's used to threshold/filter
low-confidence detections in production pipelines, not just displayed for information.

🔄 **Alternatives:** Pass a `categories_filter` argument to `recognize_pii_entities` to restrict detection to
specific PII categories only (e.g. just `Email` and `PhoneNumber`) if you don't want the model redacting
things like organization names.


In [ ]:
for idx, doc in enumerate(results):
    print(f"--- Document {idx + 1} ---")
    print(f"Redacted text: {doc.redacted_text}")
    print("Detected entities:")
    for entity in doc.entities:
        print(f"  [{entity.category}] '{entity.text}'  (confidence: {entity.confidence_score:.2f})")
    print()

## Summary

This notebook called Azure AI Language's dedicated PII detection model on a support message and printed both
the individually detected entities (with categories and confidence scores) and a ready-to-use redacted version
of the text. This is the kind of task well-suited to a purpose-built NLP endpoint rather than an LLM prompt:
deterministic categories, calibrated confidence scores, and a built-in redaction output — all without writing
or maintaining a prompt.


## Try It Yourself

1. **Easy:** Add a second document to the `documents` list containing a credit card number or physical address
   and see which PII categories the service picks up.
2. **Intermediate:** Use `categories_filter=["Email", "PhoneNumber"]` in the `recognize_pii_entities` call so
   the organization name ("Acme Logistics") is *not* redacted, only contact details.
3. **Advanced:** Write a helper that only keeps entities with `confidence_score >= 0.85`, and manually rebuilds
   a redacted string from the original text using the surviving entities' character offsets
   (`entity.offset`/`entity.length`) instead of relying on `doc.redacted_text`.
